# AIT303 Assignment 1 — BiGRU Sentiment Models

**Author:** [Your Name]
**Date:** May 2026

Trains 2 BiGRU variants on the IMDB 50K dataset with Word2Vec embeddings:
| # | Model | Embedding |
|---|-------|-----------|
| 1 | BiGRU | Word2Vec CBOW |
| 2 | BiGRU | Word2Vec Skip-Gram |

Both models are evaluated with 5-fold Stratified K-Fold cross-validation.

---
### ⚡ Colab Instructions
If running on Google Colab:
1. Upload the `data_asg/` folder to your Google Drive (as `data_asg/`)
2. Set `COLAB = True` in the config cell below
3. Run all cells — the notebook will mount your Drive and read data from it

Heavy computation cells (5-fold cross-validation for both embeddings) are marked with **⚡ HEAVY** and may take 60-90 minutes each on a T4 GPU.


In [ ]:
# ============================================
# CONFIGURATION
# ============================================
# Set to True when running on Google Colab
COLAB = True

# Data and model directories
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/data_asg'
    MODEL_DIR = f'{DATA_DIR}/models'
else:
    DATA_DIR = 'data_asg'
    MODEL_DIR = 'models'

print(f"Running in {'COLAB' if COLAB else 'LOCAL'} mode")
print(f"Data directory: {DATA_DIR}")
print(f"Model directory: {MODEL_DIR}")


## 1. Setup & Imports


In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Word embeddings
import gensim
from gensim.models import Word2Vec

# Deep learning
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Bidirectional, GRU, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Cross-validation and evaluation
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# Reproducibility
import random
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

print("All imports loaded successfully")


## 2. Load Preprocessed Data


In [ ]:
# Load preprocessed IMDB data
df = pd.read_csv(f'{DATA_DIR}/preprocessed_imdb.csv')
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df['sentiment'].value_counts())
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head(3)


In [ ]:
# Encode sentiment: positive -> 1, negative -> 0
df['sentiment_encoded'] = df['sentiment'].map({'positive': 1, 'negative': 0})
print(f"Encoded distribution:\n{df['sentiment_encoded'].value_counts()}")

# Extract lemmatized texts and labels (Phase 3 uses only lemmatized)
lemmatized_texts = df['lemmatized'].values
y = df['sentiment_encoded'].values

# NOTE: No held-out test split — all data reserved for 5-fold StratifiedKFold
print(f"Lemmatized texts shape: {lemmatized_texts.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class balance: {np.bincount(y)}")


## 3. Train Word2Vec Embeddings


### 3.1 Tokenize Lemmatized Text


In [ ]:
# Tokenize lemmatized text into list of token lists for Word2Vec
sentences = [text.split() for text in df['lemmatized']]

print(f"Number of sentences (reviews): {len(sentences)}")
print(f"First review tokens (first 20): {sentences[0][:20]}")
print(f"Average tokens per sentence: {np.mean([len(s) for s in sentences]):.1f}")


### 3.2 Train CBOW Embeddings (sg=0)


In [ ]:
# CBOW: predicts target word from context (sg=0)
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0,  # CBOW
    workers=4,
    epochs=5,
)

print(f"CBOW vocabulary size: {len(cbow_model.wv)}")
print(f"Vector shape: {cbow_model.wv.vectors.shape}")
print(f"\nCBOW 'movie' top-5 similar:")
print(cbow_model.wv.most_similar('movie', topn=5))


### 3.3 Train Skip-Gram Embeddings (sg=1)


In [ ]:
# Skip-Gram: predicts context words from target (sg=1)
sg_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,  # Skip-Gram
    workers=4,
    epochs=5,
)

print(f"Skip-Gram vocabulary size: {len(sg_model.wv)}")
print(f"Vector shape: {sg_model.wv.vectors.shape}")
print(f"\nSkip-Gram 'movie' top-5 similar:")
print(sg_model.wv.most_similar('movie', topn=5))


### 3.4 Save Word2Vec Models


In [ ]:
import os
os.makedirs(MODEL_DIR, exist_ok=True)

# Save CBOW model
cbow_model.save(f'{MODEL_DIR}/word2vec_cbow.model')
cbow_size = os.path.getsize(f'{MODEL_DIR}/word2vec_cbow.model')
print(f"Saved word2vec_cbow.model ({cbow_size/1024/1024:.1f} MB)")

# Save Skip-Gram model
sg_model.save(f'{MODEL_DIR}/word2vec_sg.model')
sg_size = os.path.getsize(f'{MODEL_DIR}/word2vec_sg.model')
print(f"Saved word2vec_sg.model ({sg_size/1024/1024:.1f} MB)")

print("\nWord2Vec models saved successfully.")


## 4. Build Vocabulary & Embedding Matrix


### 4.1 Tokenizer & Vocabulary


In [ ]:
# Fit Tokenizer on all lemmatized texts (standard for NLP CV)
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(lemmatized_texts)

word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 2  # +2 for padding (idx 0) + OOV (idx 1)
EMBEDDING_DIM = 100
MAXLEN = 200

print(f"Vocabulary size (unique words): {len(word_index)}")
print(f"VOCAB_SIZE (with pad+OOV): {VOCAB_SIZE}")
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")
print(f"MAXLEN: {MAXLEN}")


### 4.2 Build CBOW Embedding Matrix


In [ ]:
def build_embedding_matrix(w2v_model, tokenizer, vocab_size, embedding_dim):
    """Construct embedding matrix from Word2Vec weights."""
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    hit, miss = 0, 0
    for word, i in tokenizer.word_index.items():
        if i >= vocab_size:
            continue
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]
            hit += 1
        else:
            miss += 1
    coverage = hit / (hit + miss) * 100
    print(f"Coverage: {hit} found, {miss} OOV ({coverage:.1f}% hit rate)")
    return embedding_matrix

cbow_embedding_matrix = build_embedding_matrix(cbow_model, tokenizer, VOCAB_SIZE, EMBEDDING_DIM)
print(f"CBOW embedding matrix shape: {cbow_embedding_matrix.shape}")


### 4.3 Build Skip-Gram Embedding Matrix


In [ ]:
sg_embedding_matrix = build_embedding_matrix(sg_model, tokenizer, VOCAB_SIZE, EMBEDDING_DIM)
print(f"Skip-Gram embedding matrix shape: {sg_embedding_matrix.shape}")


### 4.4 Define BiGRU Model Builder


In [ ]:
def build_bigru(vocab_size, embedding_dim, embedding_matrix, maxlen=200):
    """Build 1-layer BiGRU sentiment classifier with pre-trained embeddings."""
    model = Sequential([
        Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            input_length=maxlen,
            trainable=True,
            mask_zero=True,
        ),
        Bidirectional(GRU(128)),
        Dropout(0.5),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.AUC(name='auc')],
    )
    return model


In [ ]:
# Quick sanity check — build and verify model architecture
test_model = build_bigru(VOCAB_SIZE, EMBEDDING_DIM, cbow_embedding_matrix, MAXLEN)
test_model.summary()

trainable_params = sum([tf.keras.backend.count_params(p) for p in test_model.trainable_weights])
print(f"\nTotal trainable parameters: {trainable_params:,}")
print("Model built successfully!")

# Clean up test model (memory management)
del test_model
tf.keras.backend.clear_session()
